In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scikit-learn matplotlib seaborn -q

import platform
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [2]:
# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 — 자기완결적 스냅샷
# (지난 노드에서 다룬 오염 요소를 적당히 섞어 둡니다)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 1500

regions = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n, p=[0.4, 0.25, 0.15, 0.1, 0.1])
membership = np.random.choice(["basic", "silver", "gold", "vip"], n, p=[0.5, 0.25, 0.15, 0.1])
channels = np.random.choice(["web", "app", "app ", "APP"], n, p=[0.5, 0.4, 0.05, 0.05])
categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                          p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)
amount = (prices * quantities).astype(float)

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(35, 9, n).round().astype(int),
    "region": regions,
    "membership": membership,
    "channel": channels,
    "category": categories,
    "price": prices.astype(float),
    "quantity": quantities,
    "amount": amount,
})

# 오염 심기: 결측·이상치·표기 혼재
orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan
orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan
orders.loc[5, "customer_age"] = 999             # 입력 실수성 이상치
orders.loc[8, "customer_age"] = -3              # 불가능한 음수
orders.loc[12, "quantity"] = 80                 # 비정상적으로 큰 주문
orders.loc[20, "region"] = " 서울 "             # 앞뒤 공백
orders.loc[21, "region"] = "Seoul"              # 영문 표기
orders.loc[40, "membership"] = "VIP"            # 대소문자 혼재

print("이번 달 주문 데이터 준비 완료:", orders.shape)
orders.head()

이번 달 주문 데이터 준비 완료: (1500, 9)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


In [3]:
# 예제: 같은 결과를 세 가지 방법으로 — 'amount'에 부가세 10% 더하기
amounts = orders["amount"].dropna().head(8)
print("원본:", amounts.tolist())

# (a) apply: 사용자 정의 함수 적용
def add_vat(x):
    return x * 1.1

by_apply = amounts.apply(add_vat)

# (b) map: 함수도 받지만, dict로 치환할 때 더 자연스러움
by_map = amounts.map(add_vat)

# (c) 벡터화: 함수 없이 연산자 하나
by_vec = amounts * 1.1

print("apply 결과:", by_apply.round(0).tolist())
print("map   결과:", by_map.round(0).tolist())
print("벡터화 결과:", by_vec.round(0).tolist())

원본: [749700.0, 19900.0, 19900.0, 29900.0, 59700.0, 389700.0, 19900.0, 89900.0]
apply 결과: [824670.0, 21890.0, 21890.0, 32890.0, 65670.0, 428670.0, 21890.0, 98890.0]
map   결과: [824670.0, 21890.0, 21890.0, 32890.0, 65670.0, 428670.0, 21890.0, 98890.0]
벡터화 결과: [824670.0, 21890.0, 21890.0, 32890.0, 65670.0, 428670.0, 21890.0, 98890.0]


In [4]:
# 예제: 세 가지 스케일러를 직접 비교 — 'amount'에 적용
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

amt = orders[["amount"]].dropna()   # 2D로 두는 게 sklearn 관례

mm = MinMaxScaler().fit_transform(amt)
ss = StandardScaler().fit_transform(amt)
rs = RobustScaler().fit_transform(amt)

scaled = pd.DataFrame({
    "original": amt["amount"].values,
    "minmax":   mm.flatten(),
    "standard": ss.flatten(),
    "robust":   rs.flatten(),
})
print(scaled.describe().round(2))

        original   minmax  standard   robust
count    1440.00  1440.00   1440.00  1440.00
mean    75532.57     0.09      0.00     0.51
std     95960.56     0.13      1.00     1.37
min      9900.00     0.00     -0.68    -0.43
25%     19900.00     0.01     -0.58    -0.28
50%     39800.00     0.04     -0.37     0.00
75%     89900.00     0.11      0.15     0.72
max    749700.00     1.00      7.03    10.14


In [5]:
# 예제: 'map'이 가장 어울리는 자리 — dict 치환
gender_kr_to_num = {"M": 1, "F": 0}
sample_gender = pd.Series(["M", "F", "F", "M", "F"])

# map은 dict를 그대로 받습니다. 이게 가장 자연스러움.
print("map(dict):", sample_gender.map(gender_kr_to_num).tolist())

# 같은 일을 apply로 하려면 dict.get을 래핑해야 합니다(가독성 떨어짐).
print("apply(lambda):", sample_gender.apply(lambda x: gender_kr_to_num.get(x)).tolist())

map(dict): [1, 0, 0, 1, 0]
apply(lambda): [1, 0, 0, 1, 0]


In [6]:
# 예제: 속도 비교 — 10만 건에 '부가세 10% 더하기'
big = pd.Series(np.random.randint(1000, 100_000, 100_000).astype(float))

# (a) apply
start = time.time()
_ = big.apply(lambda x: x * 1.1)
t_apply = time.time() - start

# (b) map
start = time.time()
_ = big.map(lambda x: x * 1.1)
t_map = time.time() - start

# (c) 벡터화
start = time.time()
_ = big * 1.1
t_vec = time.time() - start

print(f"apply  : {t_apply*1000:.1f} ms")
print(f"map    : {t_map*1000:.1f} ms")
print(f"벡터화 : {t_vec*1000:.1f} ms")
print(f"→ 벡터화가 apply보다 약 {t_apply/max(t_vec, 1e-9):.0f}배 빠릅니다 (환경에 따라 다름)")

apply  : 19.2 ms
map    : 18.5 ms
벡터화 : 1.0 ms
→ 벡터화가 apply보다 약 19배 빠릅니다 (환경에 따라 다름)


In [ ]:
# 예제: 모두마켓 데이터에 셋을 모두 적용 — '주문 등급' 만들기
# 규칙: amount >= 100000 → 'high', 30000 이상 → 'mid', 그 외 'low'

# (a) apply (행 단위 — 가독성은 좋지만 느림)
def classify(amount):
    if pd.isna(amount):
        return np.nan
    if amount >= 100_000:
        return "high"
    elif amount >= 30_000:
        return "mid"
    return "low"

orders["grade_apply"] = orders["amount"].apply(classify)

# (b) 벡터화 (np.select가 'if-elif-else'의 벡터화 버전)
orders["grade_vec"] = np.select(
    [orders["amount"] >= 100_000, orders["amount"] >= 30_000],
    ["high", "mid"],
    default="low"
)
# 결측은 결측으로 두기
orders.loc[orders["amount"].isna(), "grade_vec"] = np.nan

# 결과가 같은지 확인 (결측 처리도 함께)
print("두 방식 결과 동일?:", (orders["grade_apply"].fillna("X") == orders["grade_vec"].fillna("X")).all())
print(orders[["amount", "grade_apply", "grade_vec"]].head())

In [7]:
# 예제: 단계마다 함수로 빼기 — '한 함수 = 한 책임'
def clean_strings(df):
    # 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid(df, age_min=0, age_max=120, qty_max=20):
    # 불가능한 값(나이 범위·과대 수량)과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
    )

def add_features(df):
    # 분석에 쓸 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
    )

print("세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.")

세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.


In [8]:
# 예제: 한 줄로 조립
cleaned = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
)

print("원본:", orders.shape, "→ 정제 후:", cleaned.shape)
cleaned.head(3)

원본: (1500, 9) → 정제 후: (1404, 11)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,is_premium
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,13.527430,0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,9.898525,0
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,9.898525,0


In [9]:
# 예제: 인코딩/스케일링 함수 추가
from sklearn.preprocessing import RobustScaler

def encode_categories(df):
    # membership은 Ordinal, region·channel·category는 One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    # One-Hot
    out = pd.concat(
        [out,
         pd.get_dummies(out["region"], prefix="region", dtype=int),
         pd.get_dummies(out["channel"], prefix="ch", dtype=int),
         pd.get_dummies(out["category"], prefix="cat", dtype=int)],
        axis=1
    )
    return out

def scale_numeric(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 RobustScaler로 스케일링.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in cols], index=df.index)
    return pd.concat([df, scaled_df], axis=1)

# 전체 파이프라인 한 줄
pipeline_result = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
    .pipe(encode_categories)
    .pipe(scale_numeric)
)

print("최종 shape:", pipeline_result.shape)
print("새로 생긴 컬럼 일부:", [c for c in pipeline_result.columns if "scaled" in c or c.startswith("region_")][:8])

최종 shape: (1404, 27)
새로 생긴 컬럼 일부: ['region_경기', 'region_대구', 'region_부산', 'region_서울', 'region_인천', 'customer_age_scaled', 'amount_scaled', 'quantity_scaled']


In [14]:
# ─────────────────────────────────────────────
# 시나리오 0 — 원본 CSV 파일 준비 (실무에서는 외부에서 받아오는 단계)
# ─────────────────────────────────────────────
work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)
input_csv  = work_dir / "orders_raw.csv"
output_csv = work_dir / "orders_clean.csv"

orders.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, "—", input_csv.stat().st_size, "bytes")

원본 CSV 저장: d006_work\orders_raw.csv — 81901 bytes


In [11]:
# 시나리오 1 — 단계 함수들
def load_orders(path):
    # CSV에서 주문 데이터를 읽어 옵니다.
    return pd.read_csv(path)

def clean_strings_full(df):
    # 문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    # 불가능한 값과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
        .drop_duplicates()
        .reset_index(drop=True)
    )

def add_derived(df):
    # 분석용 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",
                                np.where(d["amount"] >= 30_000, "mid", "low"))
    )

def encode_full(df):
    # membership=Ordinal, 나머지=One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)

def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 Robust로 스케일링하고 _scaled 접미사를 붙입니다.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled,
                             columns=[f"{c}_scaled" for c in cols],
                             index=df.index)
    return pd.concat([df, scaled_df], axis=1)

print("단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.")

단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.


In [15]:
# 시나리오 2 — end-to-end 파이프라인
def preprocess(input_path):
    # 원본 CSV 경로를 받아 전처리된 DataFrame을 돌려줍니다.
    return (
        load_orders(input_path)
        .pipe(clean_strings_full)
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)
        .pipe(add_derived)
        .pipe(encode_full)
        .pipe(scale_with_robust)
    )

# 진짜 한 줄 호출
clean_df = preprocess(input_csv)
print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])
print("출력 컬럼 수:", clean_df.shape[1])
clean_df.head(3)

입력 행 수 → 출력 행 수: 1500 → 1404
출력 컬럼 수: 28


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,...,ch_app,ch_web,cat_가전,cat_도서,cat_뷰티,cat_식품,cat_패션,customer_age_scaled,amount_scaled,quantity_scaled
0,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,13.527430,...,1,0,1,0,0,0,0,-0.916667,10.141429,2.0
1,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,9.898525,...,0,1,0,0,0,0,1,0.833333,-0.284286,0.0
2,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,9.898525,...,1,0,0,0,0,0,1,0.500000,-0.284286,0.0


In [16]:
# 시나리오 3 — 저장 + 재로드 검증
clean_df.to_csv(output_csv, index=False)

reloaded = pd.read_csv(output_csv)
print("저장 파일:", output_csv, "—", output_csv.stat().st_size, "bytes")
print("저장 직전 shape:", clean_df.shape)
print("다시 읽은  shape:", reloaded.shape)
print("동일한가?:", clean_df.shape == reloaded.shape)

저장 파일: d006_work\orders_clean.csv — 199683 bytes
저장 직전 shape: (1404, 28)
다시 읽은  shape: (1404, 28)
동일한가?: True


In [17]:
# 품질 리포트 함수
def quality_report(before: pd.DataFrame, after: pd.DataFrame) -> dict:
    # 전처리 전·후 데이터의 품질 차이를 dict로 반환합니다.
    report = {
        "rows_before": before.shape[0],
        "rows_after":  after.shape[0],
        "removed_pct": round(100 * (1 - after.shape[0] / before.shape[0]), 2),
        "cols_before": before.shape[1],
        "cols_after":  after.shape[1],
        "new_cols":    after.shape[1] - before.shape[1],
        "missing_top5_before": (
            before.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "missing_top5_after": (
            after.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "dtypes_after": after.dtypes.value_counts().to_dict(),
    }
    return report

# 사용
rep = quality_report(orders, clean_df)
for k, v in rep.items():
    print(f"{k}: {v}")

rows_before: 1500
rows_after: 1404
removed_pct: 6.4
cols_before: 9
cols_after: 28
new_cols: 19
missing_top5_before: {'amount': 60, 'customer_age': 30, 'order_id': 0, 'membership': 0, 'region': 0}
missing_top5_after: {'order_id': 0, 'customer_age': 0, 'region': 0, 'membership': 0, 'channel': 0}
dtypes_after: {dtype('int64'): 15, dtype('float64'): 7, <StringDtype(storage='python', na_value=nan)>: 6}
